# 03: STRING Networks for pLLPS-Enriched Functional Groups

This notebook fetches STRING protein-protein interactions for functionally enriched membrane protein groups and enriches them with pLLPS scores.

**Workflow:**
1. Load pLLPS-enriched functional groups from notebook 02
2. For each enriched group, fetch high-confidence STRING interactions
3. Filter for high-confidence interactions (score ≥ 700/1000)
4. Match interactors with pLLPS scores
5. Save interaction data for visualization

**Inputs:**
- `results/pllps_enriched_groups.json`: List of enriched groups
- `results/functional_groups_with_pllps.csv`: Classified proteins with scores
- `results/full_dataset.csv`: Full dataset with pLLPS scores

**Outputs:**
- `results/string_networks_by_group/{group}_interactions.csv`: Interactions per group
- `results/enriched_group_interactions_summary.json`: Summary statistics

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import llps_functions as lf
import json
import importlib
from pathlib import Path

# Reload module
importlib.reload(lf)

# Constants
SCORE_THRESHOLD = 600  # STRING score threshold (0-1000); ~0.6 confidence cutoff
STRING_CACHE_DIR = Path('data/string_cache')
STRING_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("✅ Libraries imported successfully")
print(f"STRING score threshold: {SCORE_THRESHOLD}/1000")

✅ Libraries imported successfully
STRING score threshold: 700/1000


## 1. Load Enriched Functional Groups and pLLPS Data

In [2]:
# Load data
df_full = lf.load_analysis_result('full_dataset', format='csv')
functional_groups_df = lf.load_analysis_result('functional_groups_with_pllps', format='csv')

# Load enriched groups list
with open('results/pllps_enriched_groups.json', 'r') as f:
    enriched_data = json.load(f)
    enriched_groups = enriched_data.get('enriched_groups', enriched_data.get('high_enriched', []))

print(f"\n📊 Data Loaded:")
print(f"   Total proteins with pLLPS: {len(df_full)}")
print(f"   Classified membrane proteins: {len(functional_groups_df)}")
print(f"   Enriched groups identified: {len(enriched_groups)}")
print(f"\n   Enriched groups: {enriched_groups}")

# Create pLLPS lookup dictionary
pllps_dict = dict(zip(df_full['Entry'], df_full['p(LLPS)']))
print(f"\n   pLLPS lookup created: {len(pllps_dict)} proteins")

✅ Loaded CSV from: results/full_dataset.csv (20366 rows)
✅ Loaded CSV from: results/functional_groups_with_pllps.csv (20366 rows)

📊 Data Loaded:
   Total proteins with pLLPS: 20366
   Classified membrane proteins: 20366
   Enriched groups identified: 6

   Enriched groups: ['Structural', 'Other', 'Enzyme', 'Ion Channel', 'Transporter', 'GPCR']

   pLLPS lookup created: 20366 proteins


## 2. Fetch and Process STRING Interactions for Each Group

In [5]:
# Process each enriched group
output_dir = Path('results/string_networks_by_group')
output_dir.mkdir(parents=True, exist_ok=True)

all_interactions_summary = {}

# Filter to manageable groups
manageable_groups = ['Structural', 'Ion Channel', 'Transporter', 'GPCR']
enriched_groups_filtered = [g for g in enriched_groups if g in manageable_groups]
print(f"\n⚠️  Processing selected groups with manageable sizes:")
print(f"   Groups: {enriched_groups_filtered}")
print(f"   (Skipping 'Other' and 'Enzyme' due to size)\n")

for group_name in enriched_groups_filtered:
    print(f"\n{'='*60}")
    print(f"Processing: {group_name}")
    print(f"{'='*60}")
    
    # Get proteins in this group
    group_proteins = functional_groups_df[functional_groups_df['Functional Group'] == group_name].copy()
    protein_ids = group_proteins['Entry'].dropna().unique().tolist()
    protein_set = set(protein_ids)
    
    print(f"\n📥 Fetching interactions for {len(protein_ids)} proteins...")
    
    # Map UniProt -> STRING IDs (improves hit rate)
    string_to_uniprot, uniprot_to_string = lf.get_string_mapping(protein_ids)
    mapped_ids = [uniprot_to_string.get(pid, pid) for pid in protein_ids]
    
    # Fetch interactions from STRING (physical network, high confidence)
    interactions_df, errors = lf.fetch_string_interactions(
        protein_ids=mapped_ids,
        score_threshold=SCORE_THRESHOLD,
        network_type="physical",
        use_cache=True
    )
    
    if errors:
        print("\nNotes/Warnings:")
        for msg in errors:
            print(f"   - {msg}")
    
    if interactions_df is None or interactions_df.empty:
        print(f"   ⚠️  No interactions returned for {group_name}")
        all_interactions_summary[group_name] = {'protein_count': len(protein_ids), 'interaction_count': 0}
        continue
    
    # Normalize scores (STRING returns 0-1 for score)
    interactions_df['combined_score'] = (interactions_df['score'] * 1000).round(0)
    interactions_df['score_normalized'] = interactions_df['score']
    
    # Filter for high-confidence interactions
    interactions_df = interactions_df[interactions_df['score'] >= SCORE_THRESHOLD / 1000.0].copy()
    print(f"\n✓ Total interactions (post-filter): {len(interactions_df)}")
    
    if interactions_df.empty:
        print(f"   ⚠️  No high-confidence interactions found after filtering")
        all_interactions_summary[group_name] = {'protein_count': len(protein_ids), 'interaction_count': 0}
        continue
    
    # Map STRING IDs back to UniProt and add pLLPS
    interactions_df['protein'] = interactions_df['stringId_A'].map(string_to_uniprot)
    interactions_df['partner'] = interactions_df['stringId_B'].map(string_to_uniprot)
    
    # Fallback to preferred names if mapping missing
    interactions_df['protein'] = interactions_df['protein'].fillna(interactions_df['preferredName_A'])
    interactions_df['partner'] = interactions_df['partner'].fillna(interactions_df['preferredName_B'])
    
    # Add pLLPS values
    interactions_df['pllps_1'] = interactions_df['protein'].map(pllps_dict)
    interactions_df['pllps_2'] = interactions_df['partner'].map(pllps_dict)
    
    # Keep interactions where both partners are in the functional group and have pLLPS
    mask_in_group = interactions_df['protein'].isin(protein_set) & interactions_df['partner'].isin(protein_set)
    mask_has_pllps = interactions_df['pllps_1'].notna() & interactions_df['pllps_2'].notna()
    matched_df = interactions_df[mask_in_group & mask_has_pllps].copy()
    
    if matched_df.empty:
        print(f"   ⚠️  No interactions matched to pLLPS dataset within the group")
        all_interactions_summary[group_name] = {'protein_count': len(protein_ids), 'interaction_count': 0}
        continue
    
    # Rename columns expected by downstream visualization (notebook 05)
    matched_df['protein_name'] = matched_df['preferredName_A']
    matched_df['partner_name'] = matched_df['preferredName_B']
    
    # Save to CSV
    output_file = output_dir / f"{group_name.lower().replace(' ', '_')}_interactions.csv"
    matched_df.to_csv(output_file, index=False)
    print(f"\n✅ Saved: {output_file}")
    
    # Calculate summary statistics
    high_high = ((matched_df['pllps_1'] > 0.7) & (matched_df['pllps_2'] > 0.7)).sum()
    high_low = (
        ((matched_df['pllps_1'] > 0.7) & (matched_df['pllps_2'] <= 0.7)) |
        ((matched_df['pllps_1'] <= 0.7) & (matched_df['pllps_2'] > 0.7))
    ).sum()
    mean_partner_pllps = float(matched_df[['pllps_1', 'pllps_2']].stack().mean())
    
    all_interactions_summary[group_name] = {
        'protein_count': len(protein_ids),
        'interaction_count': len(matched_df),
        'high_high_interactions': int(high_high),
        'high_low_interactions': int(high_low),
        'mean_partner_pllps': mean_partner_pllps,
        'file': str(output_file)
    }

# Save summary
summary_file = Path('results/enriched_group_interactions_summary.json')
with open(summary_file, 'w') as f:
    json.dump(all_interactions_summary, f, indent=2)

print(f"\n{'='*60}")
print(f"✅ Summary saved: {summary_file}")
print(f"\nReady for network visualization in notebook 04 or 05!")


⚠️  Processing selected groups with manageable sizes:
   Groups: ['Structural', 'Ion Channel', 'Transporter', 'GPCR']
   (Skipping 'Other' and 'Enzyme' due to size)


Processing: Structural

📥 Fetching interactions for 1443 proteins...
Mapping 1443 proteins to STRING names in 1 batches...
  Batch 1/1: Mapped 1423 items

✓ Total interactions (post-filter): 311

✅ Saved: results/string_networks_by_group/structural_interactions.csv

Processing: Ion Channel

📥 Fetching interactions for 334 proteins...
Mapping 334 proteins to STRING names in 1 batches...
  Batch 1/1: Mapped 334 items

✓ Total interactions (post-filter): 282

✅ Saved: results/string_networks_by_group/ion_channel_interactions.csv

Processing: Transporter

📥 Fetching interactions for 313 proteins...
Mapping 313 proteins to STRING names in 1 batches...
  Batch 1/1: Mapped 312 items

✓ Total interactions (post-filter): 13

✅ Saved: results/string_networks_by_group/transporter_interactions.csv

Processing: GPCR

📥 Fetching inter